In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import pandas as pd

import sys

sys.path.append('T:\EL_experiment\Codes\CCEP_human\Python_Analysis/py_functions')

from scipy.stats import norm
from tkinter import *
import scipy
from scipy import signal

import platform
from glob import glob
from scipy.spatial import distance
import basic_func as bf
from scipy.integrate import simps
from numpy import trapz
import tqdm
from matplotlib.patches import Rectangle
from pathlib import Path
import statsmodels.api as sm
import statsmodels.formula.api as smf

#
from scipy.signal import hilbert, butter, filtfilt
import scipy.stats as stats
from tqdm.notebook import trange, tqdm
import load_summary as ls
cwd             = os.getcwd()

##all 
cond_vals   = np.arange(4)
cond_labels = ['BM', 'BL', 'Fuma', 'Benzo']
cond_colors = ['#494159','#594157', "#F1BF98","#8FB996"]
dist_groups = np.array([[0,15],[15,30],[30,5000]])
dist_labels = ['local (<15 mm)', 'short (<30mm)', 'long']
sub_path  ='X:\\4 e-Lab\\' # y:\\eLab

sys.path.append('T:\\EL_experiment\Codes\CCEP_human\Python_Analysis\py_functions')

In [6]:
import load_summary as ls
sub_path  ='X:\\4 e-Lab\\' # y:\\eLab
# df = ls.get_connections(subjs, sub_path)

subjs = ["EL004","EL005","EL010", "EL011", "EL012", "EL013", "EL014", "EL015", "EL016", "EL019", "EL020", "EL021",
         "EL022", "EL024", "EL026", "EL027", "EL028"]

In [54]:
subjs = ["EL003", "EL004","EL005","EL008","EL010", "EL011", "EL012", "EL013", "EL014", "EL015", "EL016","EL017", "EL019", "EL020", "EL021",
         "EL022", "EL024", "EL025", "EL026", "EL027", "EL028"]

In [64]:
subj ='EL003'
path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)
path_infos = os.path.join(sub_path, 'Patients', subj, 'Electrodes')
lbls = pd.read_excel(os.path.join(path_infos, subj + "_labels.xlsx"), header=0, sheet_name='BP')
if "type" in lbls.columns:
    lbls = lbls[lbls.type == 'SEEG'].reset_index(drop=True)

labels_all = lbls.label.values

## ATLAS

In [125]:
df = pd.read_excel('X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\\atlas.xlsx')

In [127]:
# Convert the DataFrame to LaTeX format and save it to a file
latex_table = df.to_latex(index=False, longtable=True)

file = 'X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\\atlas.tex'
with open(file, 'w') as file:
    file.write(latex_table)


## Patients


In [71]:
subj ='EL004'

In [ ]:
for prot in ['InputOutput', 'PairedPulse', 'BrainMapping']:
    

In [75]:
prot = 'InputOutput'

In [77]:
count_all = np.zeros((len(subjs)),)

In [79]:
subj

'EL008'

In [80]:
subjs = ["EL003","EL004","EL005","EL010", "EL011", "EL012", "EL013", "EL014", "EL015", "EL016", "EL019", "EL020", "EL021",
         "EL022", "EL024", "EL026", "EL027", "EL028"]

In [128]:
count_total = 0
prot ='BrainMapping'
for ix_subj, subj in enumerate(subjs):
    path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)
    file = os.path.join(path_patient_analysis, 'stimlist_hypnogram.csv')
    stimlist = pd.read_csv(file)
    if subj == 'EL003':
        stimlist.insert(0, 'Prot', 'PairedPulse')
    count = len(stimlist[stimlist.Prot ==prot])
    count_all[ix_subj] = count
    
count_all[count_all==0] =np.nan
print(np.nanmedian(count_all))
print(np.nanquantile(count_all, 0.25))
print(np.nanquantile(count_all, 0.75))

10999.0
6860.0
27104.5


In [132]:
count_all

array([   nan,    nan,    nan,  2247., 10344.,  3719.,  5355.,  5680.,
        8040.,  8832., 24333., 26800., 35030., 27981., 17204., 10999.,
       32402., 27409.,    nan,    nan,    nan])

In [131]:
np.nansum(count_all)

246375.0

In [112]:
# Initialize an empty DataFrame with the desired columns
df = pd.DataFrame(columns=['subj', 'n_blocks', 'Wake', 'NREM', 'REM'])

for ix_subj, subj in enumerate(subjs):
    path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)
    file = os.path.join(path_patient_analysis, 'stimlist_hypnogram.csv')
    stimlist = pd.read_csv(file)
    stimlist = add_sleepstate(stimlist)
    n_blocks = np.max(stimlist['stim_block'])
    count_total = len(stimlist)
    # Calculate counts and percentages
    count_wake = len(stimlist[stimlist['SleepState'] == 'Wake'])
    wake_info = count_wake/count_total*100 # 3f'{count_wake} ({count_wake/count_total*100:0.2f}%)'
    
    count_nrem = len(stimlist[stimlist['SleepState'] == 'NREM'])
    nrem_info = count_nrem/count_total*100 #f'{count_nrem} ({count_nrem/count_total*100:0.2f}%)'
    
    count_rem = len(stimlist[stimlist['SleepState'] == 'REM'])
    rem_info = count_rem/count_total*100 #f'{count_rem} ({count_rem/count_total*100:0.2f}%)'
    
    # Append the new row to the DataFrame
    df = df.append({
        'subj': subj,
        'n_blocks': n_blocks,
        'Wake': wake_info,
        'NREM': nrem_info,
        'REM': rem_info
    }, ignore_index=True)


In [118]:
md

67.57314007586969

In [123]:
for ss in ['Wake', 'NREM', 'REM']:
    md = np.median(df[ss])
    i2 = np.quantile(df[ss], 0.25)
    i4 = np.quantile(df[ss], 0.75)
    print(f'{md:0.2f}\% [{i2:0.2f},{i4:0.2f}]')

67.57\% [61.87,76.94]
18.81\% [15.75,25.97]
5.11\% [2.77,8.17]


In [109]:
# Initialize an empty DataFrame with the desired columns
df = pd.DataFrame(columns=['ID', '# Block', 'Wake', 'NREM', 'REM'])

for ix_subj, subj in enumerate(subjs):
    path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)
    file = os.path.join(path_patient_analysis, 'stimlist_hypnogram.csv')
    stimlist = pd.read_csv(file)
    stimlist = add_sleepstate(stimlist)
    n_blocks = np.max(stimlist['stim_block'])
    count_total = len(stimlist)
    # Calculate counts and percentages
    count_wake = len(stimlist[stimlist['SleepState'] == 'Wake'])
    wake_info = f'{count_wake} ({count_wake/count_total*100:0.2f}%)'
    
    count_nrem = len(stimlist[stimlist['SleepState'] == 'NREM'])
    nrem_info = f'{count_nrem} ({count_nrem/count_total*100:0.2f}%)'
    
    count_rem = len(stimlist[stimlist['SleepState'] == 'REM'])
    rem_info = f'{count_rem} ({count_rem/count_total*100:0.2f}%)'
    
    # Append the new row to the DataFrame
    df = df.append({
        'ID': subj,
        '# Block': n_blocks,
        'Wake': wake_info,
        'NREM': nrem_info,
        'REM': rem_info
    }, ignore_index=True)


In [111]:
# Convert the DataFrame to LaTeX format and save it to a file
latex_table = df.to_latex(index=False, longtable=True)

file = 'X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\patients\\sleep_blocks.tex'
with open(file, 'w') as file:
    file.write(latex_table)


In [99]:
def add_sleepstate(con_trial):
    if not 'SleepState' in con_trial:
        con_trial.insert(6, 'SleepState', 'Wake')
    if not 'Ictal' in con_trial:
        con_trial.insert(6, 'Ictal', 0)
    con_trial.loc[(con_trial.SleepState == 'W'), 'SleepState'] = 'Wake'
    con_trial.loc[(con_trial.sleep == 0), 'SleepState'] = 'Wake'
    con_trial.loc[(con_trial.sleep > 1) & (con_trial.sleep < 4), 'SleepState'] = 'NREM'
    con_trial.loc[(con_trial.sleep == 1), 'SleepState'] = 'NREM1'
    con_trial.loc[(con_trial.sleep == 6), 'SleepState'] = 'SZ'
    con_trial.loc[(con_trial.sleep == 4), 'SleepState'] = 'REM'
    con_trial.loc[(con_trial.sleep == 5), 'SleepState'] = 'Unknown'
    con_trial.loc[(con_trial.Ictal != 0), 'SleepState'] = 'SZ'
    return con_trial

In [74]:
prot = 'InputOutput'
count = len(stimlist[stimlist.Prot ==prot])

,ix_h,ix,Prot,StimNum,h,s,min,ChanP,Int_prob,date,sleep,stim_block
0,17.633889,0,InputOutput,0,17,2,38,38,4.0,20210126,0,1
1,17.649444,1,InputOutput,1,17,58,38,38,12.0,20210126,0,1
2,17.638333,2,InputOutput,2,17,18,38,6,6.0,20210126,0,1
3,17.638333,3,InputOutput,3,17,18,38,6,1.0,20210126,0,1
4,17.641389,4,InputOutput,4,17,29,38,42,0.2,20210126,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
18431,8.956667,18431,InputOutput,7199,8,24,57,22,3.0,20210128,0,40
18432,8.957222,18432,InputOutput,7200,8,26,57,22,4.0,20210128,0,40
18433,8.955278,18433,InputOutput,7201,8,19,57,38,0.6,20210128,0,40
18434,8.960278,18434,InputOutput,7202,8,37,57,42,3.0,20210128,0,40


In [62]:
import os
import pandas as pd
import numpy as np
import re

# Assuming 'subjs', 'sub_path', 'regions_all', and 'atlas' are already defined
# Also assuming 'bf.get_Stim_chans' is a function you have access to

arr = []
for ix_s, subj in enumerate(subjs):
    path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)
    path_infos = os.path.join(sub_path, 'Patients', subj, 'Electrodes')
    lbls = pd.read_excel(os.path.join(path_infos, subj + "_labels.xlsx"), header=0, sheet_name='BP')
    if "type" in lbls.columns:
        lbls = lbls[lbls.type == 'SEEG'].reset_index(drop=True)
        
    labels_all = lbls.label.values

    
    ix = 0
    for label in labels_all:
        region = atlas.loc[atlas['Extra-Modifier'] == " ".join(re.findall("[a-zA-Z_]+", label)), 'Region'].values
        if len(region)>0:
            if np.isin(region[0], regions_all):
                ix += 1
        else:
            region = atlas.loc[atlas['Abbreviation'] == " ".join(re.findall("[a-zA-Z_]+", label)), 'Region'].values
            if len(region)>0:
                if np.isin(region[0], regions_all):
                    ix += 1
    # Append the accumulated data for each subject to 'arr'
    arr.append([subj, ix, block_number])

# Create the DataFrame after the loop
df = pd.DataFrame(arr, columns=['Subj', 'Num GM', 'Num Block'])


In [63]:
df

,Subj,Num GM,Num Block
0,EL003,40,46
1,EL004,63,46
2,EL005,46,46
3,EL008,34,46
4,EL010,43,46
5,EL011,54,46
6,EL012,77,46
7,EL013,52,46
8,EL014,47,46
9,EL015,56,46


In [27]:
CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
tab_region = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='plot')
tab_region = tab_region.sort_values('Order').reset_index(drop=True)
regions_all = tab_region.Area.values
region_col = tab_region.color.values

CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
all_region = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='atlas')

In [49]:
CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
atlas = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='atlas')


In [51]:
import os
import pandas as pd
import numpy as np
import re

# Assuming 'subjs', 'sub_path', 'regions_all', and 'atlas' are already defined
# Also assuming 'bf.get_Stim_chans' is a function you have access to

arr = []
for ix_s, subj in enumerate(subjs):
    path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)
    path_infos = os.path.join(sub_path, 'Patients', subj, 'Electrodes')
    lbls = pd.read_excel(os.path.join(path_infos, subj + "_labels.xlsx"), header=0, sheet_name='BP')
    if "type" in lbls.columns:
        lbls = lbls[lbls.type == 'SEEG'].reset_index(drop=True)
        
    stimlist_file = os.path.join(path_patient_analysis, 'InputOutput', 'CR', 'data', 'stimlist_CR.csv')
    stimlist = pd.read_csv(stimlist_file)
    labels_all, labels_region, labels_clinic, coord_all, StimChans, StimChanSM, StimChansC, StimChanIx, stimlist = bf.get_Stim_chans(
        stimlist,
        lbls)
    block_number = len(np.unique(stimlist.stim_block))
    
    ix = 0
    for label in labels_all:
        region = atlas.loc[atlas['Extra-Modifier'] == " ".join(re.findall("[a-zA-Z_]+", label)), 'Region'].values
        if len(region)>0:
            if np.isin(region[0], regions_all):
                ix += 1
        else:
            region = atlas.loc[atlas['Abbreviation'] == " ".join(re.findall("[a-zA-Z_]+", label)), 'Region'].values
            if len(region)>0:
                if np.isin(region[0], regions_all):
                    ix += 1
    # Append the accumulated data for each subject to 'arr'
    arr.append([subj, ix, block_number])

# Create the DataFrame after the loop
df = pd.DataFrame(arr, columns=['Subj', 'Num GM', 'Num Block'])


In [53]:
df.to_excel('X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\patients\\CR_blocks.xlsx')

In [7]:
arr = []
for ix_s, subj in enumerate(subjs):
    path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)
    path_infos = os.path.join(sub_path,'Patients', subj,'Electrodes')
    lbls = pd.read_excel(os.path.join(path_infos, subj + "_labels.xlsx"), header=0, sheet_name='BP')
    if "type" in lbls.columns:
        lbls = lbls[lbls.type=='SEEG']
        lbls = lbls.reset_index(drop=True)
        
    stimlist_file = path_patient_analysis + '\\InputOutput\\CR\\data\\stimlist_CR.csv'
    stimlist = pd.read_csv(stimlist_file)
    labels_all, labels_region, labels_clinic, coord_all, StimChans, StimChanSM, StimChansC, StimChanIx, stimlist = bf.get_Stim_chans(
    stimlist,
    lbls)
    block_number = len(np.unique(stimlist.stim_block))
    
    ix = 0
    for label in labels_all:
        region = atlas.loc[atlas.Abbreviation == " ".join(re.findall("[a-zA-Z_]+",label)), 'Region'].values[0]
        if np.isin(region, regions_all):
            ix = ix+1
    arr = [[subj,ix, block_number,]]
df = pd.DataFrame(arr, columns =['Subj', 'Num GM', 'Num Block']) 